# QFabric — Overview & Run Order

**Quantum Network Emulation on FABRIC: BB84 QKD over a P4 programmable data plane.**

Alice (photon source) → BMv2 P4 switch (fiber-loss quantum channel) → Bob (detector),
with classical BB84 sifting over a real FABRIC WAN link.

## Run the notebooks in this order

| # | Notebook | What it does | Where it runs |
|---|----------|--------------|---------------|
| 0 | `00_overview` | Orientation + environment check (this notebook) | Anywhere |
| 1 | `01_setup_slice` | Provision the FABRIC slice, install BMv2, compile P4, start the switch | FABRIC JupyterHub |
| 2 | `02_run_experiment` | Run BB84 across the slice, collect results, verify | FABRIC JupyterHub |
| 3 | `03_cross_validation` | Compare QFabric (measured) vs QFabric-sim, SeQUeNCe, NetSquid — run on the slice nodes | FABRIC JupyterHub |
| 4 | `04_analysis` | Load results and generate all plots & tables | Anywhere |
| 5 | `05_run_all_scenarios` | Run **every** scenario (singles + sweeps) on the slice + sweep figures | FABRIC JupyterHub |
| 6 | `06_network_effects` | Measure classical-network (latency/jitter/loss) impact on QKD throughput | FABRIC JupyterHub |

## Prerequisites
- A FABRIC account, a project, and your tokens configured in JupyterHub (for notebooks 1–2).
- `pip install -r requirements.txt` from the project root (Python 3.11 recommended).
- For the cross-validation (notebook 3): your netsquid.org credentials in `NETSQUID_USER` / `NETSQUID_PASS` (notebook 3 builds SeQUeNCe/NetSquid/QFabric-sim envs on the slice nodes via `deploy_fabric.setup_sim_envs`). See the README.

### At a glance
- **Purpose:** orient you and confirm your environment before touching a slice.
- **Inputs:** none.
- **Outputs:** an environment report (core deps, optional simulators, fablib).
- **Runs on / runtime:** anywhere; < 1 min.
- **If something fails:** if a *core* dep is MISSING, run `pip install -r requirements.txt` (Python 3.11 recommended) and re-run.

## Environment check
Confirms required deps and reports which optional simulators / fablib are present.

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

def check(mod, required=False):
    try:
        m = importlib.import_module(mod)
        print(f'  [OK]      {mod:24s} {getattr(m, "__version__", "")}')
        return True
    except Exception as e:
        print(f'  {"[MISSING]" if required else "[optional]"} {mod:24s} ({e.__class__.__name__})')
        return False

print(f'Python {sys.version.split()[0]}  |  project: {PROJECT_DIR}\n')
print('Core (required):')
core_ok = all([check('numpy', True), check('yaml', True), check('qne', True), check('validation', True)])
print('\nOptional cross-validation backends (notebook 3):')
check('sequence'); check('netsquid')
print('\nFABRIC deployment (notebooks 1-2):')
check('fabrictestbed_extensions')
print('\nCore environment OK — continue to 01_setup_slice' if core_ok else '\nInstall core deps: pip install -e .')

## Quick smoke test (optional, runs anywhere)
Runs the pure-Python cross-validation on the baseline scenario. With no simulators installed it reports QFabric only and marks the others **SKIPPED** (never a false pass).

In [ ]:
import subprocess
out = subprocess.run([sys.executable, '-m', 'validation.compare',
                      'validation/scenarios/baseline_1km.yml'],
                     cwd=str(PROJECT_DIR), capture_output=True, text=True)
print(out.stdout or out.stderr)

---
**Next:** `01_setup_slice` to provision the FABRIC slice.